In [1]:
from pathlib import Path
import json, csv, hashlib, html
from datetime import datetime
import numpy as np
import rasterio
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.backends.backend_pdf import PdfPages

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'runs/croma_sen1floods11').is_dir())
SOURCE = ROOT / 'reports/multimodal_diagnostics/20261028_095846'
RUNS = {k: ROOT / v for k, v in json.loads((SOURCE / 'provenance.json').read_text())['runs'].items()}
HAND = ROOT / 'dataset/sen1floods11_hand_labeled_only/HandLabeled'
OUT = ROOT / 'reports/upernet_visual_comparison' / datetime.now().strftime('%Y%m%d_%H%M%S')
OUT.mkdir(parents=True, exist_ok=True)
ROWS_PER_PAGE = 4
RGB_SCALE = 3000.0
VH_MIN, VH_MAX = -30.0, -5.0  # Fixed dB display limits for every chip; no model preprocessing.

In [2]:
mask_colors = ['#29323f', '#20bde0', '#c4c7cd']
error_colors = ['#29323f', '#35b99d', '#f0ab50', '#e85d65', '#c4c7cd']
mask_cmap = ListedColormap(mask_colors)
error_cmap = ListedColormap(error_colors)
gray_cmap = plt.get_cmap('gray').copy()
gray_cmap.set_bad('#c4c7cd')
models = ['S1', 'S2', 'S1S2']
indexes = {}
for split in ['non_bol', 'bol']:
    indexes[split] = {}
    for model in models:
        paths = sorted((RUNS[model] / split).glob('*_prediction.tif'))
        mapping = {}
        for path in paths:
            chip = path.name.split('_S1Hand_')[0].split('_S2Hand_')[0]
            if chip in mapping:
                raise ValueError(f'Duplicate {model}: {chip}')
            mapping[chip] = path
        indexes[split][model] = mapping
    assert set(indexes[split]['S1']) == set(indexes[split]['S2']) == set(indexes[split]['S1S2'])
    assert len(indexes[split]['S1']) == {'non_bol':90, 'bol':15}[split]

records, manifest = [], []
def load_case(split, chip):
    label_path = HAND / 'LabelHand' / f'{chip}_LabelHand.tif'
    with rasterio.open(label_path) as src:
        y = src.read(1)
        grid = src.shape, src.crs, src.transform
    assert np.isin(y, [-1, 0, 1, 255]).all()
    valid = (y == 0) | (y == 1)
    def read_aligned(path, bands):
        with rasterio.open(path) as src:
            assert src.shape == grid[0] and src.crs == grid[1] and src.transform.almost_equals(grid[2]), str(path)
            return src.read(bands, masked=True).astype('float32').filled(np.nan)
    rgb = np.moveaxis(read_aligned(HAND / 'S2Hand' / f'{chip}_S2Hand.tif', [4,3,2]), 0, -1)
    rgb_valid = np.isfinite(rgb).all(axis=-1)
    rgb = np.clip(np.nan_to_num(rgb) / RGB_SCALE, 0, 1)
    rgb[~rgb_valid] = [0.77,0.78,0.80]
    vh = read_aligned(HAND / 'S1Hand' / f'{chip}_S1Hand.tif', 2)
    predictions, metrics = {}, {}
    for model in models:
        path = indexes[split][model][chip]
        pred = read_aligned(path, 1)
        assert np.isin(pred, [0,1]).all()
        tp = int(np.sum(valid & (y == 1) & (pred == 1)))
        fp = int(np.sum(valid & (y == 0) & (pred == 1)))
        fn = int(np.sum(valid & (y == 1) & (pred == 0)))
        iou = tp / (tp + fp + fn) if tp + fp + fn else float('nan')
        predictions[model] = pred
        metrics[model] = iou
        records.append(dict(split=split, chip=chip, model=model, valid_pixels=int(valid.sum()), tp=tp, fp=fp, fn=fn, water_iou=iou))
        manifest.append(dict(split=split,chip=chip,model=model,path=str(path.relative_to(ROOT)),sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    return rgb, vh, y, valid, predictions, metrics

In [3]:
# Every chip is included in alphabetical order, without selecting by performance.
pages = []
for split in ['non_bol', 'bol']:
    chips = sorted(indexes[split]['S1'])
    with PdfPages(OUT / f'{split}_predictions.pdf') as pred_pdf, PdfPages(OUT / f'{split}_errors.pdf') as error_pdf:
        for start in range(0, len(chips), ROWS_PER_PAGE):
            selected = chips[start:start + ROWS_PER_PAGE]
            cases = [load_case(split, chip) for chip in selected]
            for mode, pdf in [('predictions', pred_pdf), ('errors', error_pdf)]:
                fig, axes = plt.subplots(len(selected), 6, figsize=(18, 3.25 * len(selected) + 0.9), squeeze=False)
                for row, (chip, case) in enumerate(zip(selected, cases)):
                    rgb, vh, y, valid, predictions, metrics = case
                    axes[row,0].imshow(rgb, interpolation='nearest')
                    axes[row,0].set_title('S2 RGB (B4/B3/B2)', fontsize=10)
                    axes[row,0].set_ylabel(chip, fontsize=9)
                    axes[row,1].imshow(np.ma.masked_invalid(vh), cmap=gray_cmap, vmin=VH_MIN, vmax=VH_MAX, interpolation='nearest')
                    axes[row,1].set_title('S1 VH (-30 to -5 dB)', fontsize=10)
                    axes[row,2].imshow(np.where(valid, y, 2), cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
                    axes[row,2].set_title('Reference label', fontsize=10)
                    for col, model in enumerate(models, 3):
                        pred = predictions[model]
                        if mode == 'predictions':
                            display = np.where(valid, pred, 2)
                            cmap, vmax = mask_cmap, 2
                        else:
                            display = np.zeros(y.shape, dtype='uint8')
                            display[(y == 1) & (pred == 1)] = 1
                            display[(y == 0) & (pred == 1)] = 2
                            display[(y == 1) & (pred == 0)] = 3
                            display[~valid] = 4
                            cmap, vmax = error_cmap, 4
                        axes[row,col].imshow(display, cmap=cmap, vmin=0, vmax=vmax, interpolation='nearest')
                        iou = metrics[model]
                        score = f'{iou*100:.1f}%' if np.isfinite(iou) else 'undefined'
                        axes[row,col].set_title(f'{model} UPerNet | water IoU {score}', fontsize=10)
                    for ax in axes[row]:
                        ax.set_xticks([])
                        ax.set_yticks([])
                colors = mask_colors if mode == 'predictions' else error_colors
                labels = ['Non-water', 'Water', 'Ignored label'] if mode == 'predictions' else ['True negative', 'True positive', 'False positive', 'False negative', 'Ignored label']
                fig.legend(handles=[Patch(color=c, label=l) for c,l in zip(colors,labels)], loc='lower center', ncol=len(labels), frameon=False)
                page = start // ROWS_PER_PAGE + 1
                fig.suptitle(f'{split} | {mode} | page {page} | chips {start+1}-{start+len(selected)} of {len(chips)}', fontsize=13)
                fig.tight_layout(rect=(0, .035, 1, .97))
                name = f'{split}_{mode}_{page:02d}.png'
                fig.savefig(OUT / name, dpi=150)
                pdf.savefig(fig, dpi=150)
                plt.close(fig)
                pages.append(dict(split=split, mode=mode, file=name, chips=selected))
            print(f'{split}: {start+len(selected)}/{len(chips)} chips rendered', flush=True)

In [4]:
with (OUT / 'per_chip_metrics.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)
(OUT / 'prediction_manifest.json').write_text(json.dumps(manifest, indent=2))
(OUT / 'provenance.json').write_text(json.dumps(dict(runs={k:str(v.relative_to(ROOT)) for k,v in RUNS.items()}, source_diagnostics=str(SOURCE.relative_to(ROOT)), rgb_divisor=RGB_SCALE, vh_display_db=[VH_MIN,VH_MAX], selection='All 105 saved test chips; alphabetical within each split', inference=False, pages=pages), indent=2))
# Verify per-chip counts against the independently saved diagnostic audit.
with (SOURCE / 'paired_chip_diagnostics.csv').open() as f:
    expected = {(r['split'],r['chip']):r for r in csv.DictReader(f)}
for r in records:
    e = expected[(r['split'],r['chip'])]
    for metric in ['tp','fp','fn']:
        assert r[metric] == int(e[f"{r['model']}_{metric.upper()}"])
intro = '<h1>CROMA + UPerNet: S1, S2 and S1+S2</h1><p>All 105 test chips, paired on identical raster grids. Input panels show original acquisitions with display-only scaling. S1 VH is the second SAR band, not a label. S2 RGB is a visual reference, not the complete annotation procedure. Gray input pixels indicate missing/nonfinite RGB or VH; gray mask pixels are ignored reference labels. Predictions outside valid labels are hidden consistently. Undefined water IoU means no valid water union, not perfect performance. PNGs can be opened at full resolution.</p>'
links = '<p>' + ' | '.join(f'<a href="{s}_{m}.pdf">{s} {m} PDF</a>' for s in ['non_bol','bol'] for m in ['predictions','errors']) + '</p>'
body = ''
for mode in ['predictions','errors']:
    body += f'<h2>{mode.title()}</h2>'
    for p in [p for p in pages if p['mode'] == mode]:
        body += f'<details><summary>{p["split"]}: {html.escape(", ".join(p["chips"]))}</summary><a href="{p["file"]}"><img loading="lazy" src="{p["file"]}" style="width:100%"></a></details>'
(OUT / 'index.html').write_text('<!doctype html><meta charset="utf-8"><title>CROMA visual comparison</title><style>body{font-family:Arial;margin:24px;max-width:1800px}summary{cursor:pointer;padding:12px}details{border:1px solid #ccc;margin:8px 0}</style>' + intro + links + body, encoding='utf-8')
print(f'Validated all {len(records)} prediction/label pairs. Gallery: {OUT / "index.html"}')

Validated 315 prediction/label pairs. Gallery: D:\UNI\CROMA_experiment\reports\upernet_visual_comparison\20261028_101359\index.html


In [5]:

# These are the two S2 band combinations documented for the manual annotation GUI.
# Display stretch is illustrative; the paper does not specify the GUI stretch.
input_pages = []
with PdfPages(OUT / 'annotation_inputs.pdf') as pdf:
    for split in ['non_bol','bol']:
        chips = sorted(indexes[split]['S1'])
        for start in range(0, len(chips), ROWS_PER_PAGE):
            selected = chips[start:start+ROWS_PER_PAGE]
            fig, axes = plt.subplots(len(selected), 5, figsize=(16,3.7*len(selected)+.9), squeeze=False)
            for row, chip in enumerate(selected):
                with rasterio.open(HAND / 'S2Hand' / f'{chip}_S2Hand.tif') as src:
                    s2 = src.read(masked=True).astype('float32').filled(np.nan)
                with rasterio.open(HAND / 'S1Hand' / f'{chip}_S1Hand.tif') as src:
                    vh = src.read(2,masked=True)
                with rasterio.open(HAND / 'LabelHand' / f'{chip}_LabelHand.tif') as src:
                    y = src.read(1)
                for col, bands in enumerate([[4,3,2],[12,8,4],[8,11,4]]):
                    rgb = np.moveaxis(s2[np.array(bands)-1],0,-1)
                    valid_rgb = np.isfinite(rgb).all(-1)
                    rgb = np.clip(np.nan_to_num(rgb)/RGB_SCALE,0,1)
                    rgb[~valid_rgb] = [.77,.78,.80]
                    axes[row,col].imshow(rgb,interpolation='nearest')
                    axes[row,col].set_title('S2 RGB '+ '/'.join('B'+str(b) for b in bands),fontsize=10)
                axes[row,0].set_ylabel(chip,fontsize=9)
                axes[row,3].imshow(vh,cmap=gray_cmap,vmin=VH_MIN,vmax=VH_MAX,interpolation='nearest')
                axes[row,3].set_title('S1 VH (-30 to -5 dB)',fontsize=10)
                axes[row,4].imshow(np.where(np.isin(y,[0,1]),y,2),cmap=mask_cmap,vmin=0,vmax=2,interpolation='nearest')
                axes[row,4].set_title('Final hand label',fontsize=10)
                for ax in axes[row]:
                    ax.set_xticks([])
                    ax.set_yticks([])
            fig.suptitle(f'{split}: annotation input views | display stretch /3000 (not documented GUI stretch)',fontsize=12)
            fig.legend(handles=[Patch(color=c,label=l) for c,l in zip(mask_colors,['Non-water','Water','Ignored label'])],loc='lower center',ncol=3,frameon=False)
            fig.tight_layout(rect=(0,.035,1,.97), h_pad=2.0)
            name = f'{split}_annotation_inputs_{start//ROWS_PER_PAGE+1:02d}.png'
            fig.savefig(OUT/name,dpi=150)
            pdf.savefig(fig,dpi=150)
            plt.close(fig)
            input_pages.append((name,selected))
            print(f'Annotation views {split}: {start+len(selected)}/{len(chips)}',flush=True)
source_url = 'https://www.grss-ieee.org/wp-content/uploads/earthvision2020/july_stuff/webpage/papers/Bonafilia_Sen1Floods11_A_Georeferenced_Dataset_to_Train_and_Test_Deep_Learning_CVPRW_2020_paper.pdf'
extra = '<h2>Annotation input views</h2><p>The original paper describes analysts using S1 VH, S2 composites B12/B8/B4 and B8/B11/B4, and an initial S2 water classification. They corrected water areas and marked uncertain pixels as no data. These panels show the available acquisitions and final hand labels; they do not reconstruct the initial classification or exact GUI display stretch. <a href="'+source_url+'">Source: original paper, Hand Labeling</a>. <a href="annotation_inputs.pdf">All annotation input views (PDF)</a>.</p>'
for name,chips in input_pages:
    extra += f'<details><summary>{html.escape(", ".join(chips))}</summary><a href="{name}"><img loading="lazy" src="{name}" style="width:100%"></a></details>'
(OUT/'index.html').write_text((OUT/'index.html').read_text(encoding='utf-8').split('<h2>Annotation input views</h2>')[0],encoding='utf-8')
with (OUT/'index.html').open('a',encoding='utf-8') as f:
    f.write(extra)
print(OUT/'annotation_inputs.pdf')


Created annotation input views for all 105 chips.
